# Traffic vehicle detection, tracking and counting: demo

YOLOv8 detects cars and trucks, ByteTrack follows each one across frames, and every vehicle is counted once when it crosses the green line (upwards = ENTER, downwards = EXIT).

**To run it:** select **Runtime → Run all**. It takes about 2–3 minutes in total; the first cell installs the project.
For speed, switch to a GPU first: **Runtime → Change runtime type → T4 GPU** (optional, it also works on CPU, just slower).

Source code: [github.com/akhileshsuresh97/traffic-vehicle-counting](https://github.com/akhileshsuresh97/traffic-vehicle-counting)

## 1. Install

In [ ]:
!git clone -q https://github.com/akhileshsuresh97/traffic-vehicle-counting.git
%cd traffic-vehicle-counting
!pip install -q ultralytics imageio-ffmpeg

## 2. Choose a video

By default this downloads a free sample clip, [City road with traffic on a sunny day](https://mixkit.co/free-stock-video/city-road-with-traffic-on-a-sunny-day-34265/) from Mixkit (Mixkit License, free to use).
To use your own video instead, set `USE_OWN_VIDEO = True` and run the cell: an upload button appears.
The project works best with a **fixed camera looking along a road**, not drone or dashcam footage.

In [ ]:
USE_OWN_VIDEO = False
SAMPLE_URL = "https://assets.mixkit.co/videos/34265/34265-720.mp4"

if USE_OWN_VIDEO:
    from google.colab import files
    uploaded = files.upload()
    VIDEO = next(iter(uploaded))
else:
    VIDEO = "sample.mp4"
    !wget -q -O {VIDEO} "{SAMPLE_URL}"
print("Video:", VIDEO)

## 3. Count vehicles

`LINE` is the height of the counting line as a fraction of the frame (0.5 = middle, 0.9 = near the bottom).
If the line doesn't cross the road where vehicles pass, change it and run this cell again.

In [ ]:
LINE = 0.75
SECONDS = 20  # process only the first N seconds, to keep the demo quick

!python count_vehicles.py "{VIDEO}" --model yolov8s.pt --line {LINE} --max-seconds {SECONDS} --max-width 1280

## 4. Results

In [ ]:
import base64, json
from IPython.display import HTML, Image, display

counts = json.load(open("output/counts.json"))
print(f"Entered: {counts['entered']['total']}  (cars {counts['entered']['car']}, trucks {counts['entered']['truck']})")
print(f"Exited:  {counts['exited']['total']}  (cars {counts['exited']['car']}, trucks {counts['exited']['truck']})")

video_b64 = base64.b64encode(open("output/annotated.mp4", "rb").read()).decode()
display(HTML(f'<video width="800" controls autoplay muted loop src="data:video/mp4;base64,{video_b64}"></video>'))
display(Image("output/vehicle_counts.png", width=500))

The PDF report is in `output/vehicle_count_report.pdf` (open the folder icon on the left to download it).

**How the counting works:** each vehicle gets a track ID from ByteTrack and is counted at most once per direction. Overlapping truck boxes (tractor + trailer) are merged into one, the car/truck label is a majority vote over the last 15 frames, and only vehicles inside the white counting zone with a plausible size are counted. See the [README](https://github.com/akhileshsuresh97/traffic-vehicle-counting) for details.